# Power Analysis

## Overview

Power analysis determines the sample size required to detect a meaningful effect with acceptable probability. Running experiments without it risks either wasting resources on an underpowered study or collecting far more data than necessary.

**Four quantities — fix three, solve for the fourth:**

| Quantity | Symbol | Typical value |
|---|---|---|
| Significance level | α | 0.05 |
| Power | 1−β | 0.80 |
| Effect size | d or h | domain-dependent |
| Sample size | n | what we solve for |

**Effect size conventions (Cohen):**
- Small: d = 0.2, h = 0.2
- Medium: d = 0.5, h = 0.5
- Large: d = 0.8, h = 0.8

Always define the minimum detectable effect (MDE) from domain knowledge — not from a previous underpowered study.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.power import TTestIndPower, NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
import warnings; warnings.filterwarnings('ignore')

rng = np.random.default_rng(42)
print("Power analysis for ecological restoration experiments")
print("Scenario: comparing species richness between restored vs control sites")

---
## Sample Size for t-test and Proportions

In [ ]:
# Continuous outcome: species richness (t-test)
analysis_t = TTestIndPower()
# How many sites per group to detect d=0.5 with 80% power at α=0.05?
n_per_group = analysis_t.solve_power(effect_size=0.5, alpha=0.05, power=0.8,
                                      alternative='two-sided')
print(f"t-test: n={n_per_group:.1f} per group (d=0.5, power=0.8, α=0.05)")

# Sensitivity analysis: n vs effect size
effect_sizes = np.linspace(0.1, 1.0, 50)
ns_80 = [analysis_t.solve_power(e, alpha=0.05, power=0.80) for e in effect_sizes]
ns_90 = [analysis_t.solve_power(e, alpha=0.05, power=0.90) for e in effect_sizes]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(effect_sizes, ns_80, lw=2, color='steelblue', label='Power=0.80')
axes[0].plot(effect_sizes, ns_90, lw=2, color='#e74c3c', label='Power=0.90')
axes[0].axvline(0.5, color='grey', lw=1, linestyle='--', label='Medium effect')
axes[0].set_xlabel("Cohen's d"); axes[0].set_ylabel('n per group')
axes[0].set_title('Sample Size vs Effect Size (t-test)'); axes[0].legend()
axes[0].set_ylim(0, 400)

# Binary outcome: proportion of sites exceeding target richness (proportions test)
p_control, p_treatment = 0.30, 0.50   # 20pp lift
h = proportion_effectsize(p_treatment, p_control)   # Cohen's h
n_prop = NormalIndPower().solve_power(effect_size=h, alpha=0.05, power=0.80)
print(f"\nProportions test: n={n_prop:.1f} per group")
print(f"  (30% -> 50% success rate, Cohen's h={h:.3f})")

# Power vs alpha trade-off
alphas = [0.01, 0.05, 0.10]
colors_a = ['#e74c3c','steelblue','#2ecc71']
for alpha, color in zip(alphas, colors_a):
    ns = [analysis_t.solve_power(e, alpha=alpha, power=0.80) for e in effect_sizes]
    axes[1].plot(effect_sizes, ns, lw=2, color=color, label=f'α={alpha}')
axes[1].axvline(0.5, color='grey', lw=1, linestyle='--')
axes[1].set_xlabel("Cohen's d"); axes[1].set_ylabel('n per group')
axes[1].set_title('Effect of α on Required n'); axes[1].legend()
axes[1].set_ylim(0, 500)
plt.tight_layout(); plt.show()

---
## Achieved Power Given Fixed Sample Size

In [ ]:
# Given a fixed number of available sites, what effects can we detect?
n_available = 30   # only 30 sites per group available
powers = [analysis_t.solve_power(e, alpha=0.05, nobs1=n_available)
          for e in effect_sizes]
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(effect_sizes, powers, lw=2, color='steelblue')
ax.axhline(0.80, color='#e74c3c', lw=1.5, linestyle='--', label='80% power')
ax.axhline(0.50, color='grey', lw=1, linestyle=':', label='50% power')
mde_80 = effect_sizes[np.searchsorted(powers, 0.80)]
ax.axvline(mde_80, color='orange', lw=1.5, linestyle='--',
           label=f'MDE at 80% power: d={mde_80:.2f}')
ax.set_xlabel("Cohen's d"); ax.set_ylabel('Statistical power')
ax.set_title(f'Achieved Power vs Effect Size (n={n_available} per group)')
ax.legend(); ax.set_ylim(0, 1.05)
plt.tight_layout(); plt.show()
print(f"With n={n_available} per group:")
print(f"  Minimum detectable effect at 80% power: d={mde_80:.2f}")
print(f"  Power to detect d=0.3 (small): {analysis_t.solve_power(0.3, alpha=0.05, nobs1=n_available):.1%}")
print(f"  Power to detect d=0.5 (medium): {analysis_t.solve_power(0.5, alpha=0.05, nobs1=n_available):.1%}")

---
## ANOVA Power (Multiple Groups)

In [ ]:
from statsmodels.stats.power import FtestAnovaPower
anova_power = FtestAnovaPower()
# 3 treatment groups: control, low-intensity, high-intensity restoration
# f = 0.25 (medium effect for ANOVA)
n_anova = anova_power.solve_power(effect_size=0.25, alpha=0.05,
                                   power=0.80, k_groups=3)
print(f"One-way ANOVA (k=3): n={n_anova:.1f} per group (f=0.25, power=0.80)")
# Power curve for ANOVA
fs = np.linspace(0.1, 0.6, 50)
powers_3grp = [anova_power.solve_power(f, alpha=0.05, nobs=40, k_groups=3)
               for f in fs]
powers_5grp = [anova_power.solve_power(f, alpha=0.05, nobs=40, k_groups=5)
               for f in fs]
fig, ax = plt.subplots(figsize=(9,4))
ax.plot(fs, powers_3grp, lw=2, color='steelblue', label='k=3 groups')
ax.plot(fs, powers_5grp, lw=2, color='#e74c3c',   label='k=5 groups')
ax.axhline(0.80, color='grey', lw=1, linestyle='--')
ax.set_xlabel("Cohen's f"); ax.set_ylabel('Power')
ax.set_title('ANOVA Power (n=40 per group)')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Bootstrap power simulation
def bootstrap_power(n_per_group, true_delta, sigma, n_sim=500, alpha=0.05):
    significant = 0
    for _ in range(n_sim):
        ctrl = rng.normal(15, sigma, n_per_group)
        trt  = rng.normal(15 + true_delta, sigma, n_per_group)
        _, p = stats.ttest_ind(ctrl, trt)
        if p < alpha:
            significant += 1
    return significant / n_sim

print("Bootstrap power simulation (species richness, sigma=5):")
print(f"{'n/group':>8} {'delta=1':>10} {'delta=2':>10} {'delta=3':>10}")
for n in [10, 20, 30, 50]:
    p1 = bootstrap_power(n, 1, sigma=5)
    p2 = bootstrap_power(n, 2, sigma=5)
    p3 = bootstrap_power(n, 3, sigma=5)
    print(f"{n:8d} {p1:10.2f} {p2:10.2f} {p3:10.2f}")
print("\nBootstrap power: model any complex design by simulating the data-generating process")

---

## Common Pitfalls

**1. Using a previous underpowered study to estimate effect size**  
Small studies overestimate effect sizes due to the winner's curse — only significant results get published and the effect size in a significant but underpowered test is inflated. Basing power calculations on these estimates produces new studies that are also underpowered. Use domain knowledge or meta-analytic estimates instead.

**2. Treating 80% power as a sufficient default without considering consequences**  
Eighty percent power means a 20% chance of missing a real effect. For decisions with high stakes — regulatory approval, conservation interventions — 90% or 95% power may be appropriate. Always consider the cost asymmetry between Type I and Type II errors in your domain.

**3. Computing power after the experiment to "explain" a non-significant result**  
Post-hoc power calculated from an observed non-significant result is mathematically equivalent to a function of the p-value and contains no additional information. A non-significant result does not mean the effect is zero — it means the data are consistent with a range of effect sizes. Use confidence intervals instead.

**4. Not accounting for attrition, non-compliance, or clustering in sample size**  
Power calculations for independent observations underestimate the required sample size when observations are clustered (sites within catchments), when some units drop out, or when treatment compliance is imperfect. Inflate by 1/(1−attrition rate) and apply a design effect for clustering.

**5. Setting the MDE to detect the smallest imaginable effect rather than the smallest practically relevant effect**  
Power-seeking experiments may be designed to detect effects so small they have no practical significance (e.g. 0.1 species richness improvement across hundreds of sites). Always define the MDE as the smallest effect that would actually change a decision, not simply the smallest detectable effect.
---
*python_methods_library - Samantha McGarrigle*